Step 1: Install Dependencies

In [1]:
# Cell 1: Install required libraries
!pip install opencv-python dlib gradio numpy pandas

Step 2: Download the Pre-trained Landmark Model

In [2]:
# Cell 2: Download and decompress the Dlib shape predictor model
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -dk shape_predictor_68_face_landmarks.dat.bz2
print("Dlib model downloaded and extracted successfully!")

--2026-07-13 16:29:28--  http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Resolving dlib.net (dlib.net)... 107.180.26.78
Connecting to dlib.net (dlib.net)|107.180.26.78|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 [following]
--2026-07-13 16:29:28--  https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Connecting to dlib.net (dlib.net)|107.180.26.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64040097 (61M)
Saving to: ‘shape_predictor_68_face_landmarks.dat.bz2’

shape_predictor_68_ 100%[===================>]  61.07M  45.4MB/s    in 1.3s    

2026-07-13 16:29:30 (45.4 MB/s) - ‘shape_predictor_68_face_landmarks.dat.bz2’ saved [64040097/64040097]

Dlib model downloaded and extracted successfully!


Step 3: Complete Unified Application Code

In [5]:
# Cell 3: Main Application Logic with 3-Column Single-Screen Layout
import os
import cv2
import dlib
import json
import numpy as np
import pandas as pd
import gradio as gr

# Initialize Dlib Computer Vision tools
face_detector = dlib.get_frontal_face_detector()
landmark_predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# Unified Knowledge Base synthesized from Sources 1, 2, 4, 5, and 6
PHYSIOGNOMY_RULES = {
    "face": {
        "long": "Wood-Shaped Architecture: Indicates profound strategic planning capabilities, high persistence, methodical processing, and a practical mindset. Prone to being overworked.",
        "oval": "Charming & Diplomatic: Expresses an emotionally balanced, sweet, and highly adaptive character. Possesses artistic refinement and cognitive poise.",
        "round": "Water-Shaped Architecture: Reflects radiant emotional warmth, deep sensitivity, a caring nature, and a strong relationship-oriented mindset."
    },
    "nose": {
        "long": "Commanding Power: Signifies strong individual determination, high self-confidence, proud nature, and natural executive leadership parameters.",
        "short": "Operational Flexibility: Reflects high spontaneity, quick decision-making metrics, and operational agility in rapid environments."
    },
    "eyes": {
        "large": "Luminous & Expansive: Denotes profound expressiveness, high spiritual empathy, independent creative thinking, and a communicative disposition.",
        "small": "Concentrated Focus: Denotes sharp analytical prowess, intense powers of concentration, deep observation focus, and a shrewd, detail-oriented mind."
    }
}

def extract_and_analyze_face(input_image):
    if input_image is None:
        return None, "🔮 Please upload a clear face image to begin the reading."

    image = cv2.cvtColor(input_image, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_detector(gray)

    if len(faces) == 0:
        return None, "❌ Error: AI could not resolve facial boundaries. Please ensure your face is well-lit and centered."

    shape = landmark_predictor(gray, faces[0])
    landmarks = np.array([[shape.part(i).x, shape.part(i).y] for i in range(68)])

    # Compute baseline geometric distances
    face_width = float(landmarks[16][0] - landmarks[0][0])
    face_height = float(landmarks[8][1] - landmarks[27][1])
    nose_length = float(landmarks[33][1] - landmarks[27][1])
    left_eye_width = float(landmarks[39][0] - landmarks[36][0])
    right_eye_width = float(landmarks[45][0] - landmarks[42][0])
    eye_width_avg = (left_eye_width + right_eye_width) / 2.0

    if face_width <= 0 or face_height <= 0:
        return None, "❌ Error: Invalid facial geometry metrics calculated."

    face_ratio = face_height / face_width
    nose_ratio = nose_length / face_height
    eye_ratio = eye_width_avg / face_width

    if face_ratio >= 1.40:
        face_type = "long"
    elif face_ratio <= 1.18:
        face_type = "round"
    else:
        face_type = "oval"

    nose_type = "long" if nose_ratio >= 0.38 else "short"
    eye_type = "large" if eye_ratio >= 0.24 else "small"

    report = (
        f"✨ AI REPORT ✨\n"
        f"━━━━━━━━━━━━━━━━━━━━\n\n"
        f"🎯 [Structure]: {face_type.upper()} FACE\n"
        f"📖 {PHYSIOGNOMY_RULES['face'][face_type]}\n\n"
        f"👃 [Nasal]: {nose_type.upper()} NOSE\n"
        f"📖 {PHYSIOGNOMY_RULES['nose'][nose_type]}\n\n"
        f"👁️ [Ocular]: {eye_type.upper()} EYES\n"
        f"📖 {PHYSIOGNOMY_RULES['eyes'][eye_type]}"
    )

    # Cyan/Neon tracking overlay
    annotated_image = input_image.copy()
    for (x, y) in landmarks:
        cv2.circle(annotated_image, (x, y), 3, (0, 255, 242), -1)

    return annotated_image, report

# Custom CSS (Kept exactly identical to your theme preferences, optimized margins)
custom_css = """
.gradio-container {
    background: linear-gradient(135deg, #090714 0%, #130f26 50%, #05030a 100%) !important;
    font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
    padding-bottom: 20px !important;
}
.custom-header {
    text-align: center;
    margin-bottom: 1rem;
    padding: 10px;
    border-bottom: 1px solid rgba(212, 175, 55, 0.2);
}
.custom-header h1 {
    color: #d4af37 !important;
    font-size: 2.2rem !important;
    font-weight: 800 !important;
    text-shadow: 0 0 15px rgba(212, 175, 55, 0.4);
    margin-bottom: 0.2rem;
}
.custom-header p {
    color: #a3a1cc !important;
    font-size: 1rem;
}
.gr-box, .gr-panel, .block {
    background-color: rgba(25, 20, 46, 0.6) !important;
    border: 1px solid rgba(212, 175, 55, 0.15) !important;
    border-radius: 12px !important;
    backdrop-filter: blur(8px);
    box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.37) !important;
}
.gr-button-primary {
    background: linear-gradient(90deg, #d4af37 0%, #aa841b 100%) !important;
    color: #090714 !important;
    font-weight: bold !important;
    border: none !important;
    box-shadow: 0 0 15px rgba(212, 175, 55, 0.3) !important;
    transition: all 0.3s ease !important;
}
.gr-button-primary:hover {
    transform: translateY(-2px);
    box-shadow: 0 0 25px rgba(212, 175, 55, 0.6) !important;
}
footer {
    display: none !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Default(primary_hue="amber", neutral_hue="slate"),
    css=custom_css
) as app:

    gr.HTML(
        """
        <div class="custom-header">
            <h1>🌌 LAKSHANA.AI</h1>
            <p>Ancient Vedic Samudrik Shastra Analytics Meets Modern Computer Vision</p>
        </div>
        """
    )

    # Restructured into a clean 3-Column horizontal row
    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(
                label="1. Upload Profile Photo",
                type="numpy"
            )
            submit_btn = gr.Button(
                "⚡ Run Analysis",
                variant="primary"
            )

        with gr.Column(scale=1):
            output_img = gr.Image(
                label="2. Neural Landmark Map"
            )

        with gr.Column(scale=1):
            output_txt = gr.Textbox(
                label="3. Shastra Interpretation",
                lines=14,
                interactive=False
            )

    submit_btn.click(
        fn=extract_and_analyze_face,
        inputs=[input_img],
        outputs=[output_img, output_txt]
    )

app.launch(inline=True, share=True)

/tmp/ipykernel_2292/711266400.py:136: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cd156b9c35a7691aa6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
